# 06 - Evidence semantic search index (SPEC section 8.4)
Creates the Vector Search endpoint if it is missing, then creates or syncs a Delta Sync index (`evidence_chunks_index`) on `gold_evidence_chunks`. The index uses managed embeddings from `embedding_endpoint`. The source table must have Change Data Feed enabled.

In [0]:
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("vector_search_endpoint", "")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
VS_ENDPOINT = dbutils.widgets.get("vector_search_endpoint")
EMBED = dbutils.widgets.get("embedding_endpoint")
assert CATALOG and SCHEMA and VS_ENDPOINT
SOURCE = f"{CATALOG}.{SCHEMA}.gold_evidence_chunks"
INDEX = f"{CATALOG}.{SCHEMA}.evidence_chunks_index"

In [0]:
import datetime
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

if VS_ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
vsc.wait_for_endpoint(VS_ENDPOINT, verbose=True)

cdf = spark.sql(f"SHOW TBLPROPERTIES {SOURCE}").filter("key = 'delta.enableChangeDataFeed'").collect()
assert cdf and cdf[0].value == "true", f"{SOURCE} needs delta.enableChangeDataFeed = true"

existing = [i["name"] for i in vsc.list_indexes(VS_ENDPOINT).get("vector_indexes", [])]
if INDEX in existing:
    idx = vsc.get_index(VS_ENDPOINT, INDEX)
    if idx.describe().get("status", {}).get("ready"):
        idx.sync()  # otherwise the initial sync is still running; just wait for it
else:
    idx = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT, index_name=INDEX, source_table_name=SOURCE,
        pipeline_type="TRIGGERED", primary_key="chunk_id",
        embedding_source_column="chunk_text", embedding_model_endpoint_name=EMBED)
idx.wait_until_ready(verbose=True, timeout=datetime.timedelta(hours=1))

In [0]:
res = idx.similarity_search(query_text="transfer of assets to a new company before administrators were appointed",
                            columns=["chunk_id", "evidence_id", "case_id", "doc_type", "chunk_text"], num_results=5)
rows = res.get("result", {}).get("data_array", [])
assert rows, "semantic search returned no rows"
for r in rows:
    print(r[1], r[2], r[3], "|", r[4][:120].replace("\n", " "))